# DAVE Documents API — Search & RAG Routes

**Prerequisite:** run `00_auth_setup.ipynb` first.

Both routes proxy requests to internal services (Elasticsearch for search,
a Chroma + LLM stack for RAG). They will return a 500/502 if those services
are not running.

| Method | Path | Description |
|--------|------|-------------|
| POST | `/api/search/faceted` | Faceted full-text document search via Elasticsearch |
| POST (GET params) | `/api/rag/generate` | RAG-augmented text generation |

In [ ]:
import sys, os, json, requests

sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
from auth_state import API_BASE, auth_headers

print(f"API base: {API_BASE}")

---
## POST /api/search/faceted — Faceted document search

The search term and collection filter are **query parameters**; facet filters
and pagination options go in the **request body**.

> **Note:** requires the Elasticsearch indexer service (`API_INDEXER` env var)
> to be running. Returns 500 if not configured, 502 if unreachable.

In [ ]:
# ── basic full-text search ────────────────────────────────────────────────────
resp = requests.post(
    f"{API_BASE}/search/faceted",
    params={
        "text":         "Napoleon",   # free-text query
        "collectionId": "",           # leave empty to search all collections
    },
    json={
        "metadata":    [],
        "annotations": [],
        "page":        1,
        "limit":       10,
        "isAnonymized": True,
        "n_facets":    20,
    },
    headers=auth_headers(),
)

if resp.status_code in (500, 502):
    print(f"Search service unavailable: {resp.status_code} — {resp.json().get('message')}")
else:
    resp.raise_for_status()
    result = resp.json()
    pagination = result.get("pagination", {})
    hits       = result.get("hits", [])
    print(f"Total hits: {pagination.get('total_hits')}")
    print(f"Page {pagination.get('current_page')} / {pagination.get('total_pages')}")
    for h in hits[:3]:
        print(f"  id={h.get('id')}  name={h.get('name')}")

In [ ]:
# ── search with entity annotation filter ─────────────────────────────────────
# Filter to documents that contain a 'Location' entity with value 'Paris'
resp = requests.post(
    f"{API_BASE}/search/faceted",
    params={"text": ""},
    json={
        "metadata":    [],
        "annotations": [{"type": "Location", "value": "Paris"}],
        "page":        1,
        "limit":       10,
        "isAnonymized": True,
        "n_facets":    10,
    },
    headers=auth_headers(),
)

if resp.status_code in (500, 502):
    print(f"Search service unavailable: {resp.status_code} — {resp.json().get('message')}")
else:
    resp.raise_for_status()
    result = resp.json()
    print(f"Total hits: {result.get('pagination', {}).get('total_hits')}")
    # Show available facet groups
    for fg in result.get("facets", {}).get("annotations", [])[:3]:
        print(f"  facet key={fg.get('key')}  doc_count={fg.get('doc_count')}")

In [ ]:
# ── search with metadata filter ───────────────────────────────────────────────
resp = requests.post(
    f"{API_BASE}/search/faceted",
    params={"text": ""},
    json={
        "metadata":    [{"type": "author", "value": "John Doe"}],
        "annotations": [],
        "page":        1,
        "limit":       10,
        "isAnonymized": False,   # search de-anonymized text
        "n_facets":    20,
    },
    headers=auth_headers(),
)

if resp.status_code in (500, 502):
    print(f"Search service unavailable: {resp.status_code} — {resp.json().get('message')}")
else:
    resp.raise_for_status()
    result = resp.json()
    print(f"Total hits: {result.get('pagination', {}).get('total_hits')}")

---
## POST /api/rag/generate — RAG-augmented generation

All parameters are **query parameters** (not body). The server:
1. Retrieves the most relevant document chunks from the vector store
2. Injects them into the system prompt
3. Calls the LLM and returns the answer

> **Note:** requires `API_LLM` (LLM endpoint) and the Chroma vector service
> to be configured. Returns 500 if env vars are missing.

In [ ]:
# ── simple non-streamed RAG query ─────────────────────────────────────────────
resp = requests.post(
    f"{API_BASE}/rag/generate",
    params={
        "text":              "Who was Napoleon Bonaparte?",
        "collectionId":      "",      # leave empty to search all collections
        "stream":            "false", # JSON response; easier to work with
        "temperature":       0.7,
        "max_tokens":        512,
        "top_p":             0.65,
        "frequency_penalty": 1.15,
        "n_results":         5,
        "force_rag":         "true",
    },
    json={},   # required empty body
    headers=auth_headers(),
)

if resp.status_code in (500, 502):
    print(f"RAG service unavailable: {resp.status_code} — {resp.json().get('message')}")
else:
    resp.raise_for_status()
    result = resp.json()
    print("Model  :", result.get("model"))
    print("Usage  :", result.get("usage"))
    print()
    print("Answer :")
    print(result.get("content"))

In [ ]:
# ── RAG with multi-turn conversation history ──────────────────────────────────
import json as _json

history = [
    {"role": "user",      "content": "Tell me about Napoleon."},
    {"role": "assistant", "content": "Napoleon Bonaparte was a French military leader..."},
]

resp = requests.post(
    f"{API_BASE}/rag/generate",
    params={
        "text":     "When did he become Emperor?",
        "messages": _json.dumps(history),  # JSON-encoded conversation history
        "stream":   "false",
        "n_results": 5,
    },
    json={},
    headers=auth_headers(),
)

if resp.status_code in (500, 502):
    print(f"RAG service unavailable: {resp.status_code} — {resp.json().get('message')}")
else:
    resp.raise_for_status()
    result = resp.json()
    print(result.get("content"))

In [ ]:
# ── RAG with streaming response ───────────────────────────────────────────────
# Streaming returns plain-text chunks; we print them as they arrive.
resp = requests.post(
    f"{API_BASE}/rag/generate",
    params={
        "text":   "Summarise Napoleon's military campaigns.",
        "stream": "true",
    },
    json={},
    headers=auth_headers(),
    stream=True,
)

if resp.status_code in (500, 502):
    print(f"RAG service unavailable: {resp.status_code}")
else:
    resp.raise_for_status()
    print("Streaming answer:")
    for chunk in resp.iter_content(chunk_size=None):
        if chunk:
            print(chunk.decode("utf-8"), end="", flush=True)
    print()  # newline after stream ends

In [ ]:
# ── RAG restricted to a specific collection ───────────────────────────────────
# Substitute with a real collection ID from 02_collections.ipynb
COLLECTION_ID = None  # e.g. "6645b3c2e4f0d123456789ab"

if not COLLECTION_ID:
    print("Set COLLECTION_ID to target a specific collection.")
else:
    resp = requests.post(
        f"{API_BASE}/rag/generate",
        params={
            "text":         "What are the main topics in this collection?",
            "collectionId": COLLECTION_ID,
            "stream":       "false",
            "n_results":    10,
        },
        json={},
        headers=auth_headers(),
    )
    if resp.status_code in (500, 502):
        print(f"RAG service unavailable: {resp.status_code}")
    else:
        resp.raise_for_status()
        print(resp.json().get("content"))